# 0. Import library

In [5]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"  # 물리적 2번 GPU만 보이게 설정

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Current device: {device}")
# 이제 torch.cuda.get_device_name(0)을 하면 물리적 2번 GPU의 이름이 나옵니다.

Current device: cuda


In [6]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset


import numpy as np
from tqdm import tqdm
import pickle
import seaborn as sns
import matplotlib.pyplot as plt
import time
import copy
from moving_average import moving_average_1d

import importlib
import policy
importlib.reload(policy)
from policy import PolicyNN

from nn_functions import surrogate

import sys
sys.path.append('../1_model')
from TiDE import TideModule, quantile_loss  


# 1-1. Import Data 

In [10]:
df_all = pd.read_csv('../0_data/merged_df_2_99_temp_depth.csv')
print(df_all.shape)
print(df_all.columns)

nan_rows = df_all[df_all.isna().any(axis=1)]

df_all = df_all.dropna()

loc_X = df_all["X"].to_numpy().reshape(-1,1)
loc_Y = df_all["Y"].to_numpy().reshape(-1,1)
loc_Z = df_all["Z"].to_numpy().reshape(-1,1)
dist_X = df_all["Dist_to_nearest_X"].to_numpy().reshape(-1,1)
dist_Y = df_all["Dist_to_nearest_Y"].to_numpy().reshape(-1,1)
# dist_Z = df_all["Dist_to_nearest_Z"].to_numpy()[::2].reshape(-1,1)
scan_spd = df_all["scanning_speed"].to_numpy().reshape(-1,1)
laser_power = df_all["Laser_power"].to_numpy().reshape(-1,1)
laser_on_off = df_all["laser_power_number"].to_numpy().reshape(-1,1)

# apply moving average for mp temp
mp_temp_raw = df_all["melt_pool_temperature"].to_numpy()
mp_temp_mv = moving_average_1d(mp_temp_raw,4)
mp_temp = copy.deepcopy(mp_temp_raw)
mp_temp[1:-2] = mp_temp_mv
mp_temp = mp_temp.reshape(-1,1)

# apply moving average for mp depth
mp_depth_raw = df_all["melt_pool_depth"].to_numpy()
mp_depth_mv = moving_average_1d(mp_depth_raw,4)
mp_depth = copy.deepcopy(mp_depth_raw)
mp_depth[1:-2] = mp_depth_mv
mp_depth = mp_depth.reshape(-1,1)       


(610615, 12)
Index(['time_index', 'melt_pool_temperature', 'melt_pool_depth',
       'scanning_speed', 'X', 'Y', 'Z', 'Dist_to_nearest_X',
       'Dist_to_nearest_Y', 'Dist_to_nearest_Z', 'Laser_power',
       'laser_power_number'],
      dtype='object')


## 1-2. Normalize data

In [13]:
# stack input array
x_original_scale = np.concatenate((loc_Z, dist_X, dist_Y, laser_power), axis=1)
y_original_scale = np.concatenate((mp_temp, mp_depth), axis=1)

# scaling
x_max = np.max(x_original_scale,0).reshape(1,-1)
x_min = np.min(x_original_scale,0).reshape(1,-1) 
y_max = np.max(y_original_scale,0).reshape(1,-1) 
y_min = np.min(y_original_scale,0).reshape(1,-1)

print("x_max:", np.round(x_max, 3).tolist())
print("x_min:", np.round(x_min, 3).tolist())
print("y_max:", np.round(y_max, 3).tolist())
print("y_min:", np.round(y_min, 3).tolist())

x_max: [[7.5, 20.0, 20.0, 732.298]]
x_min: [[0.0, 0.75, 0.75, 504.26]]
y_max: [[4509.855, 0.551]]
y_min: [[436.608, -0.559]]


In [15]:
class scalers():
    def __init__(self,x_max, x_min, y_max, y_min) -> None:
        self.x_max = x_max
        self.x_min = x_min
        self.y_max = y_max
        self.y_min = y_min
        
        return None
    
    def scaler_x(self, x_original, dim_id = -1):
        if dim_id == -1:
            x_s = -1 + 2 * ((x_original - self.x_min) / (self.x_max-self.x_min))
            return x_s
        else: 
            x_s = -1 + 2 * (x_original - self.x_min[0,dim_id]) / (self.x_max[0,dim_id] - self.x_min[0,dim_id])
            return x_s
    
    def inv_scaler_x(self, x_s, dim_id = -1):
        
        if dim_id == -1:
            x_original = (x_s + 1)*0.5*(self.x_max-self.x_min) + self.x_min
            return x_original
        else: 
            x_original = (x_s + 1)*0.5*(self.x_max[0,dim_id] - self.x_min[0,dim_id]) + self.x_min[0,dim_id]
            return x_original
        
    def scaler_y(self, y_original):
        return -1 + 2 * ((y_original - self.y_min) / (self.y_max-self.y_min))
    
    def inv_scaler_y(self, y_s):
        return (y_s + 1)*0.5*(self.y_max-self.y_min) + self.y_min

In [16]:
scaler = scalers(x_max, x_min, y_max, y_min)

x_s = scaler.scaler_x(x_original_scale)
y_s = scaler.scaler_y(y_original_scale)

print("x_s range:", np.min(x_s), "to", np.max(x_s))
print("y_s range:", np.min(y_s), "to", np.max(y_s))

print("x_s shape:", x_s.shape)
print("y_s shape:", y_s.shape)

x_s range: -1.0 to 1.0
y_s range: -1.0 to 1.0
x_s shape: (610417, 4)
y_s shape: (610417, 2)


## 1-3. Generate data

In [17]:
length = y_s.shape[0]

y_s_ref = np.random.uniform(0.0, 1.0, size=(length, 1))

e = 0.0001
y_depth_low = np.random.uniform(0.1423-e, 0.1423+e, size = (length,1))
y_depth_up = np.random.uniform(0.4126-e, 0.4126+e, size = (length,1))
# constraints!
#y_depth_low = 0*np.random.uniform(0.0, 0.5, size=(length, 1))
#y_depth_up = 0*np.random.uniform(0.5, 1.0, size=(length, 1))

y_s_const = np.concatenate((y_depth_low, y_depth_up), axis=1)

print("y_s_ref shape:", y_s_ref.shape)
print("y_s_const shape:", y_s_const.shape)

y_s_ref shape: (610417, 1)
y_s_const shape: (610417, 2)


## 1-4. Split data

In [18]:
import numpy as np
import pandas as pd
import torch
import copy
from tqdm import tqdm

# =====================================================
# 1. Load & preprocess raw data
# =====================================================
df_all = pd.read_csv('../0_data/merged_df_2_99_temp_depth.csv')

print(df_all.shape)
print(df_all.columns)

# ---- global time index (핵심) ----
df_all = df_all.reset_index(drop=True)
df_all["time_id"] = np.arange(len(df_all))

# ---- NaN handling (time_id 유지) ----
nan_rows = df_all[df_all.isna().any(axis=1)]
df_all = df_all.dropna().reset_index(drop=True)

# =====================================================
# 2. Feature extraction
# =====================================================
loc_Z = df_all["Z"].to_numpy().reshape(-1, 1)
dist_X = df_all["Dist_to_nearest_X"].to_numpy().reshape(-1, 1)
dist_Y = df_all["Dist_to_nearest_Y"].to_numpy().reshape(-1, 1)
laser_power = df_all["Laser_power"].to_numpy().reshape(-1, 1)

# ---- global time id ----
time_id_all = df_all["time_id"].to_numpy()

# =====================================================
# 3. Melt pool smoothing
# =====================================================
mp_temp_raw = df_all["melt_pool_temperature"].to_numpy()
mp_temp_mv = moving_average_1d(mp_temp_raw, 4)
mp_temp = copy.deepcopy(mp_temp_raw)
mp_temp[1:-2] = mp_temp_mv
mp_temp = mp_temp.reshape(-1, 1)

mp_depth_raw = df_all["melt_pool_depth"].to_numpy()
mp_depth_mv = moving_average_1d(mp_depth_raw, 4)
mp_depth = copy.deepcopy(mp_depth_raw)
mp_depth[1:-2] = mp_depth_mv
mp_depth = mp_depth.reshape(-1, 1)

# =====================================================
# 4. Stack original-scale arrays
# =====================================================
x_original = np.concatenate((loc_Z, dist_X, dist_Y, laser_power), axis=1)  # (N,4)
y_original = np.concatenate((mp_temp, mp_depth), axis=1)                  # (N,2)

# =====================================================
# 5. Scaling (-1 ~ 1)
# =====================================================
x_max = np.max(x_original, axis=0, keepdims=True)
x_min = np.min(x_original, axis=0, keepdims=True)
y_max = np.max(y_original, axis=0, keepdims=True)
y_min = np.min(y_original, axis=0, keepdims=True)

scaler = scalers(x_max, x_min, y_max, y_min)

x_s = scaler.scaler_x(x_original)
y_s = scaler.scaler_y(y_original)

# =====================================================
# 6. Reference / Constraint (global arrays)
# =====================================================
length = y_s.shape[0]

# =====================================================
# Global reference (temperature)
# =====================================================
y_ref_global = y_s[:, 0:1]   # (N, 1)


# depth constraint (global)
e = 1e-4
y_depth_low = np.random.uniform(0.1423 - e, 0.1423 + e, size=(length, 1))
y_depth_up  = np.random.uniform(0.4126 - e, 0.4126 + e, size=(length, 1))
y_const_global = np.concatenate((y_depth_low, y_depth_up), axis=1)  # (N,2)

# =====================================================
# 7. Train / Validation split (time-consistent)
# =====================================================
cutoff_index = int(np.round(0.8 * length))

x_train, y_train = x_s[:cutoff_index], y_s[:cutoff_index]
x_val,   y_val   = x_s[cutoff_index:], y_s[cutoff_index:]

time_id_train = time_id_all[:cutoff_index]
time_id_val   = time_id_all[cutoff_index:]

# =====================================================
# 7. (삭제/변경) Train/Val split을 먼저 하지 말고
#    8번에서 전체 샘플을 먼저 만든 뒤 shuffle split
# =====================================================

window = 50
P = 50
N = length

# 전체에서 만들 수 있는 샘플 개수
# i는 "현재 시점(start of future)" 인덱스
# i는 [window, N-P) 범위
n_all = (N - P) - window

x_past_all = np.empty((n_all, window, 4))
y_past_all = np.empty((n_all, window, 2))
x_future_all = np.empty((n_all, P, 3))
y_ref_all_seq = np.empty((n_all, P, 1))
y_const_all_seq = np.empty((n_all, P, 2))
time_idx_all_seq = np.empty(n_all, dtype=np.int64)

for i in tqdm(range(window, N - P)):
    j = i - window

    x_past_all[j] = x_s[i-window:i]
    y_past_all[j] = y_s[i-window:i]
    x_future_all[j] = x_s[i:i+P, :3]

    # reference
    scalar_shift = 0
    y_ref_all_seq[j] = y_ref_global[i:i+P] + scalar_shift

    # constraint
    y_const_all_seq[j] = y_const_global[i:i+P]

    # time index (글로벌)
    time_idx_all_seq[j] = time_id_all[i]

# =====================================================
# 8. Shuffle 후 8:2 split (샘플 단위)
# =====================================================
rng = np.random.default_rng(42)  # 재현성
perm = rng.permutation(n_all)

cut = int(np.round(0.8 * n_all))
train_idx = perm[:cut]
val_idx   = perm[cut:]

x_past_train = x_past_all[train_idx]
y_past_train = y_past_all[train_idx]
x_future_train = x_future_all[train_idx]
y_ref_train_seq = y_ref_all_seq[train_idx]
y_const_train_seq = y_const_all_seq[train_idx]
time_idx_train = time_idx_all_seq[train_idx]

x_past_val = x_past_all[val_idx]
y_past_val = y_past_all[val_idx]
x_future_val = x_future_all[val_idx]
y_ref_val_seq = y_ref_all_seq[val_idx]
y_const_val_seq = y_const_all_seq[val_idx]
time_idx_val = time_idx_all_seq[val_idx]

# =====================================================
# 9. Torch tensor conversion
# =====================================================
x_past_train = torch.tensor(x_past_train, dtype=torch.float32)
y_past_train = torch.tensor(y_past_train, dtype=torch.float32)
x_future_train = torch.tensor(x_future_train, dtype=torch.float32)
y_ref_train_seq = torch.tensor(y_ref_train_seq, dtype=torch.float32)
y_const_train_seq = torch.tensor(y_const_train_seq, dtype=torch.float32)
time_idx_train = torch.tensor(time_idx_train, dtype=torch.long)

x_past_val = torch.tensor(x_past_val, dtype=torch.float32)
y_past_val = torch.tensor(y_past_val, dtype=torch.float32)
x_future_val = torch.tensor(x_future_val, dtype=torch.float32)
y_ref_val_seq = torch.tensor(y_ref_val_seq, dtype=torch.float32)
y_const_val_seq = torch.tensor(y_const_val_seq, dtype=torch.float32)
time_idx_val = torch.tensor(time_idx_val, dtype=torch.long)


# =====================================================
# 10. Shape check
# =====================================================
print("Train shapes:")
print("x_past:", x_past_train.shape)
print("y_past:", y_past_train.shape)
print("x_future:", x_future_train.shape)
print("y_ref:", y_ref_train_seq.shape)
print("y_const:", y_const_train_seq.shape)
print("time_idx:", time_idx_train.shape)

print("\nVal shapes:")
print("x_past:", x_past_val.shape)
print("y_past:", y_past_val.shape)
print("x_future:", x_future_val.shape)
print("y_ref:", y_ref_val_seq.shape)
print("y_const:", y_const_val_seq.shape)
print("time_idx:", time_idx_val.shape)


(610615, 12)
Index(['time_index', 'melt_pool_temperature', 'melt_pool_depth',
       'scanning_speed', 'X', 'Y', 'Z', 'Dist_to_nearest_X',
       'Dist_to_nearest_Y', 'Dist_to_nearest_Z', 'Laser_power',
       'laser_power_number'],
      dtype='object')


100%|██████████| 610317/610317 [00:03<00:00, 180292.60it/s]


Train shapes:
x_past: torch.Size([488254, 50, 4])
y_past: torch.Size([488254, 50, 2])
x_future: torch.Size([488254, 50, 3])
y_ref: torch.Size([488254, 50, 1])
y_const: torch.Size([488254, 50, 2])
time_idx: torch.Size([488254])

Val shapes:
x_past: torch.Size([122063, 50, 4])
y_past: torch.Size([122063, 50, 2])
x_future: torch.Size([122063, 50, 3])
y_ref: torch.Size([122063, 50, 1])
y_const: torch.Size([122063, 50, 2])
time_idx: torch.Size([122063])


In [19]:
batch_size = 256

# ------------------ Training loader ------------------
train_dataset = TensorDataset(
    x_past_train,
    y_past_train,
    x_future_train,
    y_ref_train_seq,
    y_const_train_seq,
    time_idx_train          # ⭐ 반드시 포함
)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True           # ⭐ 핵심 수정
)

# ------------------ Validation loader ------------------
val_dataset = TensorDataset(
    x_past_val,
    y_past_val,
    x_future_val,
    y_ref_val_seq,
    y_const_val_seq,
    time_idx_val
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)


# 3. Import TiDE and Policy Model

In [20]:
# import torch
# import pickle

# # Load model
# with open('TiDE_params_single_track_square_MV_temp_depth_less_cov_0915_w50_p50.pkl', 'rb') as file:
#     nominal_params = pickle.load(file)

# TiDE = nominal_params['model'].to(device)
# total_params = sum(p.numel() for p in TiDE.parameters())

import os
import torch
import pickle
import io

# 1. GPU 설정 (커널 리스타트 후 첫 셀에서 실행 권장)
os.environ["CUDA_VISIBLE_DEVICES"] = "1" 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"현재 사용 중인 장치: {device}")

# 2. 안전한 언피클러 정의 (무조건 CPU로 먼저 로드)
class Safe_Unpickler(pickle.Unpickler):
    def find_class(self, module, name):
        if module == 'torch.storage' and name == '_load_from_bytes':
            # 파일에 어떤 장치가 기록되어 있든 상관없이 우선 CPU로 읽어들임
            return lambda b: torch.load(io.BytesIO(b), map_location='cpu')
        else:
            return super().find_class(module, name)

# 3. 모델 로드 (에러 발생 지점)
with open('TiDE_params_single_track_square_MV_temp_depth_less_cov_0915_w50_p50.pkl', 'rb') as file:
    # Safe_Unpickler를 사용하여 CPU로 로드
    nominal_params = Safe_Unpickler(file).load()

# 4. 이제 로드된 모델만 GPU(2번)로 이동
TiDE = nominal_params['model'].to(device)
total_params = sum(p.numel() for p in TiDE.parameters())

print(f"성공: 모델이 CPU를 거쳐 {device}로 안전하게 로드되었습니다.")
print(f"Total parameters: {total_params}")

현재 사용 중인 장치: cuda
성공: 모델이 CPU를 거쳐 cuda로 안전하게 로드되었습니다.
Total parameters: 796594


In [ ]:
import torch
from policy import PolicyNN

# ================================
# 1. DEVICE
# ================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ================================
# 2. CONSTANTS
# ================================
window = 50
P = 50

model_path = "/home/ftk3187/github/DPC_research/02_DED/8_policy_0302/models_saved_draft/or_3L_1024H_s1_c10_ep1000_roll100_m51200.0_case_**_83aug_lower_layer_constraint_off_uconst_trackingboostatfirstlayers(10)/policy_model_best.pth"
#model_path = "/home/ftk3187/github/DPC_research/02_DED/8_policy_0302/models_saved_draft/or_3L_1024H_s1_c10_ep1000_roll100_m51200.0_case_85aug_lower_layer_constraint_off_uconst_trackingboostatfirstlayers(10)_0406/policy_init_epoch0000.pth"
# ================================
# 3. MODEL DEFINE
# ================================
model = PolicyNN(
    past_input_dim=6,
    future_input_dim=6,
    output_dim=1,
    p=P,
    window=window,
    hidden_dim=1024,
    n_layers=3,
    dropout_p=0.0
).to(device)

# ================================
# 4. LOAD (핵심)
# ================================
ckpt = torch.load(model_path, map_location=device)

# 🔥 케이스 1: state_dict만 저장된 경우
if isinstance(ckpt, dict) and "model_state_dict" not in ckpt:
    model.load_state_dict(ckpt)

# 🔥 케이스 2: dict 안에 들어있는 경우
elif isinstance(ckpt, dict) and "model_state_dict" in ckpt:
    model.load_state_dict(ckpt["model_state_dict"])

else:
    raise ValueError("❌ 알 수 없는 checkpoint 형식")

model.eval()

print("✅ policy model loaded")

Using device: cuda
✅ policy model loaded


# 4. Rollout

In [21]:
import random
import copy
import torch
from torch.utils.data import Dataset

# =========================
# Aug samples wrapper
# =========================
class ListDataset(Dataset):
    """
    aug_samples: list of tuples
      (x_past, y_past, x_future, y_ref, y_const, time_idx)
    """
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


# =========================
# Build mapping: time_idx -> base sample index
# =========================
def build_time_to_sample_index(time_idx_tensor: torch.Tensor):
    """
    time_idx_tensor: shape (n_train,), dtype long
    returns: dict { int(time_idx) : int(sample_idx) }
    """
    time_to_j = {}
    # time_idx_tensor is on CPU typically; ensure it is
    t = time_idx_tensor.detach().cpu().tolist()
    for j, tid in enumerate(t):
        time_to_j[int(tid)] = int(j)
    return time_to_j


@torch.no_grad()
def rollout_k_step_augment_m(
    policy_model,
    tide_model,
    # base tensors (windowed training arrays)
    x_past_train,         # (N, window, 4)
    y_past_train,         # (N, window, 2)
    x_future_train,       # (N, P, 3)
    y_ref_train_seq,      # (N, P, 1)
    y_const_train_seq,    # (N, P, 2)
    time_idx_train,       # (N,)
    time_to_j,            # dict: time_idx -> j
    m: int,
    device,
    step: int = 50,       # ✅ 추가: 몇 step 전진할지 (원하면 50)
    max_tries_factor: int = 20,
):
    """
    K-step rollout augmentation.

    - 기존 1-step 대신 step개를 한 번에 적용해서 past(window)를 step만큼 전진시킴.
    - 다음 샘플의 future/ref/const는 time_idx 기준으로 t_cur + step 에서 가져옴.

    Returns:
      new_samples: list of length <= m
        (x_past_new, y_past_new, x_future_next, y_ref_next, y_const_next, time_idx_next)
      All returned tensors are on CPU.
    """

    policy_model.eval()

    n, window, _ = x_past_train.shape
    P = x_future_train.shape[1]

    if n == 0 or m <= 0:
        return []

    # step sanity
    if step <= 0:
        raise ValueError(f"step must be positive, got {step}")
    if step > P:
        raise ValueError(f"step ({step}) cannot exceed P ({P})")
    if step > window:
        raise ValueError(f"step ({step}) cannot exceed window ({window}) in this implementation")

    new_samples = []
    tries = 0
    max_tries = max_tries_factor * m

    while len(new_samples) < m and tries < max_tries:
        tries += 1

        # 1) pick a base sample index j
        j = random.randrange(n)

        # current time id (global-ish)
        t_cur = int(time_idx_train[j].item())
        t_next = t_cur + step

        # 2) need to be able to fetch the "next-time" future sequences
        if t_next not in time_to_j:
            continue
        j_next = time_to_j[t_next] 

        # 3) gather current sample tensors
        x_past = x_past_train[j:j+1].to(device)          # (1, w, 4)
        y_past = y_past_train[j:j+1].to(device)          # (1, w, 2)

        x_future = x_future_train[j:j+1].to(device)      # (1, P, 3)
        y_ref = y_ref_train_seq[j:j+1].to(device)        # (1, P, 1)
        y_const = y_const_train_seq[j:j+1].to(device)    # (1, P, 2)

        # 4) policy forward (same as training)
        policy_in_past = torch.cat((x_past, y_past), dim=2)             # (1, w, 6)
        policy_in_future = torch.cat((x_future, y_ref, y_const), dim=2) # (1, P, 6)
        u_seq = policy_model((policy_in_past, policy_in_future))        # (1, P, 1)

        # 5) TiDE forward (same as training)
        x_future_tide = torch.cat((x_future, u_seq), dim=2)             # (1, P, 4)
        past_cov = torch.cat((y_past, x_past), dim=2)                   # (1, w, 6)
        tide_pred = tide_model((past_cov, x_future_tide, None))

        # 6) take step-step outputs (t ... t+step-1)
        # control for first 'step' steps
        u_step = u_seq[:, :step, :]                                     # (1, step, 1)

        # predicted y for first 'step' steps
        # NOTE: assumes tide_pred has shape compatible with .median(dim=-1).values -> (1, P, 2)
        yhat_step = tide_pred.median(dim=-1).values[:, :step, :]        # (1, step, 2)

        # 7) build x for these step steps (covariates 3 + control 1)
        x_step = torch.cat((x_future[:, :step, :], u_step), dim=2)      # (1, step, 4)

        # 8) shift window by 'step' and append the whole step block
        #    if step==window, this effectively replaces the whole window
        x_past_new = torch.cat((x_past[:, step:, :], x_step), dim=1)    # (1, w, 4)
        y_past_new = torch.cat((y_past[:, step:, :], yhat_step), dim=1) # (1, w, 2)

        # 9) future sequences for t_next: fetch from the base arrays using j_next
        x_future_next = x_future_train[j_next].detach().cpu()           # (P, 3)
        y_ref_next = y_ref_train_seq[j_next].detach().cpu()             # (P, 1)
        y_const_next = y_const_train_seq[j_next].detach().cpu()         # (P, 2)

        # 10) package new sample (CPU tensors)
        new_samples.append((
            x_past_new.squeeze(0).detach().cpu(),     # (w, 4)
            y_past_new.squeeze(0).detach().cpu(),     # (w, 2)
            x_future_next,                            # (P, 3)
            y_ref_next,                               # (P, 1)
            y_const_next,                             # (P, 2)
            torch.tensor(t_next, dtype=torch.long)    # scalar long
        ))

    return new_samples


In [22]:
# ================================
# 1. time_idx → index 매핑
# ================================
time_to_j = build_time_to_sample_index(time_idx_train)

# ================================
# 2. rollout (10개 샘플)
# ================================
aug_samples = rollout_k_step_augment_m(
    policy_model=model,          # ← 너가 로드한 policy 모델
    tide_model=TiDE,       # ← 기존 TiDE 모델

    x_past_train=x_past_train,
    y_past_train=y_past_train,
    x_future_train=x_future_train,
    y_ref_train_seq=y_ref_train_seq,
    y_const_train_seq=y_const_train_seq,
    time_idx_train=time_idx_train,
    time_to_j=time_to_j,

    m=30,                # 🔥 10개 샘플 생성
    device=device,
    step=50              # window 길이만큼 rollout
)

print(f"✅ 생성된 샘플 개수: {len(aug_samples)}")

NameError: name 'model' is not defined

In [23]:
import numpy as np
import matplotlib.pyplot as plt
import torch

STEP = 50
N_ROLLS = 4
P = 50

def to_np(t):
    return t.cpu().numpy() if torch.is_tensor(t) else np.asarray(t)

# =========================
# 글로벌 time 매핑 (shuffle 전 전체 데이터 기준)
# =========================
time_to_global = {int(tid): i for i, tid in enumerate(time_id_all)}

def get_future_from_global(t_start):
    """t_start 시점부터 P개의 future 데이터를 글로벌 배열에서 가져오기"""
    if t_start not in time_to_global:
        return None, None, None
    g_idx = time_to_global[t_start]
    if g_idx + P > len(x_s):
        return None, None, None
    x_future = torch.tensor(x_s[g_idx:g_idx+P, :3],        dtype=torch.float32)
    y_ref    = torch.tensor(y_s[g_idx:g_idx+P, 0:1],        dtype=torch.float32)
    y_const  = torch.tensor(y_const_global[g_idx:g_idx+P],  dtype=torch.float32)
    return x_future, y_ref, y_const

# =========================
# 1st rollout (원본 데이터에서 샘플링)
# =========================
aug_samples = rollout_k_step_augment_m(
    policy_model=model,
    tide_model=TiDE,
    x_past_train=x_past_train,
    y_past_train=y_past_train,
    x_future_train=x_future_train,
    y_ref_train_seq=y_ref_train_seq,
    y_const_train_seq=y_const_train_seq,
    time_idx_train=time_idx_train,
    time_to_j=time_to_j,
    m=30,
    device=device,
    step=STEP
)
print(f"✅ 1st rollout: {len(aug_samples)}")

# =========================
# 2nd ~ 4th rollout
# =========================
all_aug_samples = [aug_samples]

for k in range(N_ROLLS - 1):
    prev = all_aug_samples[-1]
    next_samples = []

    for idx, s in enumerate(prev):

        if s is None:          # ← 추가
            next_samples.append(None)
            continue
        x_past_s   = s[0].unsqueeze(0).to(device)
        y_past_s   = s[1].unsqueeze(0).to(device)
        x_future_s = s[2].unsqueeze(0).to(device)
        y_ref_s    = s[3].unsqueeze(0).to(device)
        y_const_s  = s[4].unsqueeze(0).to(device)
        t_cur      = int(s[5].item())
        t_next     = t_cur + STEP

        # ✅ policy future = t_next 기준 (t_cur+50~t_cur+100)
        x_future_for_policy, y_ref_for_policy, y_const_for_policy = get_future_from_global(t_next)
        if x_future_for_policy is None:
            next_samples.append(None)
            continue

        with torch.no_grad():
            policy_in_past   = torch.cat((x_past_s, y_past_s), dim=2)
            policy_in_future = torch.cat((
                x_future_for_policy.unsqueeze(0).to(device),
                y_ref_for_policy.unsqueeze(0).to(device),
                y_const_for_policy.unsqueeze(0).to(device)
            ), dim=2)
            u_seq = model((policy_in_past, policy_in_future))

            # TiDE future = s[2] 그대로 (현재 구간 covariate)
            x_future_tide = torch.cat((x_future_s, u_seq), dim=2)
            past_cov      = torch.cat((y_past_s, x_past_s), dim=2)
            tide_pred     = TiDE((past_cov, x_future_tide, None))

        u_step     = u_seq[:, :STEP, :]
        yhat_step  = tide_pred.median(dim=-1).values[:, :STEP, :]
        x_step     = torch.cat((x_future_s[:, :STEP, :], u_step), dim=2)
        x_past_new = torch.cat((x_past_s[:, STEP:, :], x_step), dim=1)
        y_past_new = torch.cat((y_past_s[:, STEP:, :], yhat_step), dim=1)

        # next sample future = t_next+STEP 기준
        x_future_next, y_ref_next, y_const_next = get_future_from_global(t_next + STEP)
        if x_future_next is None:
            next_samples.append(None)
            continue

        next_samples.append((
            x_past_new.squeeze(0).detach().cpu(),
            y_past_new.squeeze(0).detach().cpu(),
            x_future_next, y_ref_next, y_const_next,
            torch.tensor(t_next, dtype=torch.long)
        ))

    all_aug_samples.append(next_samples)
    print(f"✅ {k+2}nd rollout: {len(next_samples)}")

# =========================
# 매핑 생성
# =========================
time_to_nth = [
    {int(s[5].item()) - STEP: s for s in all_aug_samples[n] if s is not None}
    for n in range(1, N_ROLLS)
]

# =========================
# loc-z 기준 정렬
# =========================
samples_with_locz = sorted(
    [(to_np(s[0])[0, 0], s) for s in aug_samples],
    key=lambda x: x[0]
)

depth_lower_const = 0.1423
depth_upper_const = 0.4126
ordinals = ['1st', '2nd', '3rd', '4th', '5th']

# =========================
# Plot
# =========================
for i, (loc_z, sample) in enumerate(samples_with_locz):

    x_past_aug, y_past_aug, _, _, _, time_idx_next = sample
    x_past_aug = to_np(x_past_aug)
    y_past_aug = to_np(y_past_aug)

    t_next = int(time_idx_next.item())
    t_cur  = t_next - STEP
    if t_cur not in time_to_j:
        continue

    j_orig      = time_to_j[t_cur]
    x_past_orig = to_np(x_past_train[j_orig])
    y_past_orig = to_np(y_past_train[j_orig])
    y_ref_orig  = to_np(y_ref_train_seq[j_orig])
    u_past_orig = x_past_orig[:, -1]

    with torch.no_grad():
        pin_past = torch.cat((x_past_train[j_orig:j_orig+1].to(device),
                              y_past_train[j_orig:j_orig+1].to(device)), dim=2)
        pin_fut  = torch.cat((x_future_train[j_orig:j_orig+1].to(device),
                              y_ref_train_seq[j_orig:j_orig+1].to(device),
                              y_const_train_seq[j_orig:j_orig+1].to(device)), dim=2)
        u_future_orig = model((pin_past, pin_fut)).cpu().numpy()[0, :, 0]

    # rollout chain 수집
    rollout_chain = [sample]
    cur_t = t_next
    for mapping in time_to_nth:
        if cur_t in mapping:
            s = mapping[cur_t]
            rollout_chain.append(s)
            cur_t = int(s[5].item())
        else:
            break
    n_cols = len(rollout_chain)

    fig, axes = plt.subplots(3, n_cols, figsize=(6*n_cols, 5),
                             sharey='row',
                             gridspec_kw={'hspace': 0.1, 'wspace': 0.3})
    if n_cols == 1:
        axes = axes[:, np.newaxis]

    for col, s in enumerate(rollout_chain):
        x_past_s = to_np(s[0])
        y_past_s = to_np(s[1])
        y_ref_s  = to_np(s[3])
        u_past_s = x_past_s[:, -1]

        t_cur_s = col * STEP
        t_prev  = np.arange(t_cur_s,        t_cur_s + STEP)
        t_cur_a = np.arange(t_cur_s + STEP, t_cur_s + STEP*2)
        t_bound = np.arange(t_cur_s,        t_cur_s + STEP*2)

        if col == 0:
            y_prev_temp  = y_past_orig[:, 0]
            y_prev_depth = y_past_orig[:, 1]
            u_prev       = u_past_orig
        else:
            prev_s       = rollout_chain[col-1]
            y_prev_temp  = to_np(prev_s[1])[:, 0]
            y_prev_depth = to_np(prev_s[1])[:, 1]
            u_prev       = to_np(prev_s[0])[:, -1]

        ord_label  = ordinals[col]
        prev_label = 'orig' if col == 0 else ordinals[col-1]

        # TEMP
        ax = axes[0, col]
        ax.plot(t_prev,  y_prev_temp,    label=f'{prev_label} past')
        ax.plot(t_cur_a, y_past_s[:, 0], label=f'{ord_label} past')
        ax.plot(t_cur_a, y_ref_s[:, 0],  '--', label=f'{ord_label} ref')
        ax.axvline(x=t_cur_s+STEP, color='gray', linestyle='--', linewidth=0.8)
        ax.set_title(f'Temp ({ord_label})')
        ax.legend(fontsize=7)
        ax.grid()
        ax.tick_params(labelbottom=False)

        # DEPTH
        ax = axes[1, col]
        ax.plot(t_prev,  y_prev_depth,   label=f'{prev_label} past')
        ax.plot(t_cur_a, y_past_s[:, 1], label=f'{ord_label} past')
        ax.plot(t_bound, [depth_lower_const]*len(t_bound), 'k--', label='lower')
        ax.plot(t_bound, [depth_upper_const]*len(t_bound), 'r--', label='upper')
        ax.axvline(x=t_cur_s+STEP, color='gray', linestyle='--', linewidth=0.8)
        ax.set_title(f'Depth ({ord_label})')
        ax.legend(fontsize=7)
        ax.grid()
        ax.tick_params(labelbottom=False)

        # LASER
        ax = axes[2, col]
        ax.plot(t_prev,  u_prev,   label=f'{prev_label} past')
        if col == 0:
            ax.plot(t_cur_a, u_future_orig[:STEP], '--', label='orig rollout')
        ax.plot(t_cur_a, u_past_s, label=f'{ord_label} past')
        ax.axvline(x=t_cur_s+STEP, color='gray', linestyle='--', linewidth=0.8)
        ax.set_title(f'Laser ({ord_label})')
        ax.legend(fontsize=7)
        ax.grid()
        ax.set_xlabel('Time Step')

    plt.suptitle(f'Sample {i} | loc-z={loc_z:.3f} | t_cur={t_cur} | n_rolls={n_cols}')
    plt.tight_layout()
    plt.show()

NameError: name 'model' is not defined

In [ ]:
s1 = aug_samples[1]
t_next_1 = int(s1[5].item())
t_cur_1 = t_next_1 - 50
print(f"1st aug: t_cur={t_cur_1}, t_next={t_next_1}")
print(f"y_past (s[1]) 구간: {t_cur_1} ~ {t_next_1}")

# 2nd rollout 첫번째
s2 = all_aug_samples[1][1]
if s2 is not None:
    t_next_2 = int(s2[5].item())
    t_cur_2 = t_next_2 - 50
    print(f"2nd aug: t_cur={t_cur_2}, t_next={t_next_2}")
    print(f"y_past (s2[1]) 구간: {t_cur_2} ~ {t_next_2}")

NameError: name 'aug_samples' is not defined

In [9]:
import numpy as np
import matplotlib.pyplot as plt
import torch

STEP = 50

# =========================
# 1. loc-z 기준 정렬
# =========================
samples_with_locz = []

for sample in aug_samples:
    x_past_aug = sample[0]
    x_past_aug_np = x_past_aug.cpu().numpy() if torch.is_tensor(x_past_aug) else np.asarray(x_past_aug)

    loc_z = x_past_aug_np[0, 0]   # 첫 timestep, 첫 feature
    samples_with_locz.append((loc_z, sample))

# loc-z 기준 정렬
samples_with_locz.sort(key=lambda x: x[0])

# =========================
# 2. plot
# =========================
for i, (loc_z, sample) in enumerate(samples_with_locz):

    x_past_aug, y_past_aug, x_future_next, y_ref_next, y_const_next, time_idx_next = sample

    # =========================
    # numpy 변환
    # =========================
    x_past_aug = x_past_aug.cpu().numpy() if torch.is_tensor(x_past_aug) else np.asarray(x_past_aug)
    y_past_aug = y_past_aug.cpu().numpy() if torch.is_tensor(y_past_aug) else np.asarray(y_past_aug)
    y_ref_next = y_ref_next.cpu().numpy() if torch.is_tensor(y_ref_next) else np.asarray(y_ref_next)

    t_next = int(time_idx_next.item())
    t_cur = t_next - STEP

    if t_cur not in time_to_j:
        continue

    j_orig = time_to_j[t_cur]

    # =========================
    # original data
    # =========================
    x_past_orig = x_past_train[j_orig].cpu().numpy()
    y_past_orig = y_past_train[j_orig].cpu().numpy()
    y_ref_orig = y_ref_train_seq[j_orig].cpu().numpy()

    # =========================
    # laser
    # =========================
    u_past_orig = x_past_orig[:, -1]
    u_past_aug = x_past_aug[:, -1]

    x_future_orig = x_future_train[j_orig:j_orig+1].to(device)
    y_ref_orig_t = y_ref_train_seq[j_orig:j_orig+1].to(device)
    y_const_orig_t = y_const_train_seq[j_orig:j_orig+1].to(device)
    x_past_orig_t = x_past_train[j_orig:j_orig+1].to(device)
    y_past_orig_t = y_past_train[j_orig:j_orig+1].to(device)

    with torch.no_grad():
        policy_in_past = torch.cat((x_past_orig_t, y_past_orig_t), dim=2)
        policy_in_future = torch.cat((x_future_orig, y_ref_orig_t, y_const_orig_t), dim=2)
        u_seq_orig = model((policy_in_past, policy_in_future))

    u_future_orig = u_seq_orig.cpu().numpy()[0, :, 0]

    # =========================
    # 시간축
    # =========================
    t_orig_past = np.arange(0, 50)
    t_orig_future = np.arange(50, 100)

    t_aug_past = np.arange(50, 100)
    t_aug_future = np.arange(100, 150)

    fig, axes = plt.subplots(1, 3, figsize=(10,2.5))

    # =========================
    # TEMP
    # =========================
    ax = axes[0]
    ax.plot(t_orig_past, y_past_orig[:,0], label='orig past')
    ax.plot(t_orig_future, y_ref_orig[:,0], '--', label='orig ref')

    ax.plot(t_aug_past, y_past_aug[:,0], label='aug past')
    ax.plot(t_aug_future, y_ref_next[:,0], '--', label='aug ref')

    ax.set_title("Temperature")
    ax.legend()
    ax.grid()

    # =========================
    # DEPTH
    # =========================
    ax = axes[1]

    depth_lower_const = 0.1423
    depth_upper_const = 0.4126

    ax.plot(t_orig_past, y_past_orig[:,1], label='orig past')
    ax.plot(t_aug_past, y_past_aug[:,1], label='aug past')

    t_full = np.arange(0,150)
    ax.plot(t_full, [depth_lower_const]*len(t_full), 'k--', label='lower')
    ax.plot(t_full, [depth_upper_const]*len(t_full), 'r--', label='upper')

    ax.set_title("Depth")
    ax.legend()
    ax.grid()

    # =========================
    # LASER
    # =========================
    ax = axes[2]
    ax.plot(t_orig_past, u_past_orig, label='orig past')
    ax.plot(t_orig_future, u_future_orig, '--', label='orig rollout')

    ax.plot(t_aug_past, u_past_aug, label='aug past')

    ax.set_title("Laser Power")
    ax.legend()
    ax.grid()







    # 🔥 loc-z 포함
    plt.suptitle(f"Sample {i} | loc-z={loc_z:.3f} | t_cur={t_cur}")
    plt.tight_layout()
    plt.show()

NameError: name 'aug_samples' is not defined

In [ ]:
aug_samples_2nd = []

for s in aug_samples:
    x_past_s   = s[0].unsqueeze(0).to(device)
    y_past_s   = s[1].unsqueeze(0).to(device)
    x_future_s = s[2].unsqueeze(0).to(device)
    y_ref_s    = s[3].unsqueeze(0).to(device)
    y_const_s  = s[4].unsqueeze(0).to(device)
    t_cur      = int(s[5].item())
    t_next     = t_cur + 50

    with torch.no_grad():
        policy_in_past   = torch.cat((x_past_s, y_past_s), dim=2)
        policy_in_future = torch.cat((x_future_s, y_ref_s, y_const_s), dim=2)
        u_seq = model((policy_in_past, policy_in_future))

        x_future_tide = torch.cat((x_future_s, u_seq), dim=2)
        past_cov      = torch.cat((y_past_s, x_past_s), dim=2)
        tide_pred     = TiDE((past_cov, x_future_tide, None))

    u_step    = u_seq[:, :50, :]
    yhat_step = tide_pred.median(dim=-1).values[:, :50, :]
    x_step    = torch.cat((x_future_s[:, :50, :], u_step), dim=2)

    x_past_new = torch.cat((x_past_s[:, 50:, :], x_step), dim=1)
    y_past_new = torch.cat((y_past_s[:, 50:, :], yhat_step), dim=1)

    # ✅ future 를 t_next 기준으로 원본에서 가져오기
    if t_next not in time_to_j:
        aug_samples_2nd.append((
            x_past_new.squeeze(0).detach().cpu(),
            y_past_new.squeeze(0).detach().cpu(),
            s[2].detach().cpu(),
            s[3].detach().cpu(),
            s[4].detach().cpu(),
            torch.tensor(t_next, dtype=torch.long)
        ))
        continue

    j_next        = time_to_j[t_next]
    x_future_next = x_future_train[j_next].detach().cpu()
    y_ref_next    = y_ref_train_seq[j_next].detach().cpu()
    y_const_next  = y_const_train_seq[j_next].detach().cpu()

    aug_samples_2nd.append((
        x_past_new.squeeze(0).detach().cpu(),
        y_past_new.squeeze(0).detach().cpu(),
        x_future_next,   # ✅ t_next 기준 future
        y_ref_next,
        y_const_next,
        torch.tensor(t_next, dtype=torch.long)
    ))

print(f"✅ 2nd rollout 샘플 개수: {len(aug_samples_2nd)}")

✅ 2nd rollout 샘플 개수: 30


In [24]:
import numpy as np
import matplotlib.pyplot as plt
import torch

STEP = 50

# 1:1 매핑
time_to_2nd = {int(s[5].item()) - 50: s for s in aug_samples_2nd}

# =========================
# 1. loc-z 기준 정렬
# =========================
samples_with_locz = []
for sample in aug_samples:
    x_past_aug_np = sample[0].cpu().numpy() if torch.is_tensor(sample[0]) else np.asarray(sample[0])
    loc_z = x_past_aug_np[0, 0]
    samples_with_locz.append((loc_z, sample))
samples_with_locz.sort(key=lambda x: x[0])

# =========================
# 2. plot
# =========================
for i, (loc_z, sample) in enumerate(samples_with_locz):

    x_past_aug, y_past_aug, x_future_next, y_ref_next, y_const_next, time_idx_next = sample

    x_past_aug = x_past_aug.cpu().numpy() if torch.is_tensor(x_past_aug) else np.asarray(x_past_aug)
    y_past_aug = y_past_aug.cpu().numpy() if torch.is_tensor(y_past_aug) else np.asarray(y_past_aug)
    y_ref_next = y_ref_next.cpu().numpy() if torch.is_tensor(y_ref_next) else np.asarray(y_ref_next)

    t_next = int(time_idx_next.item())
    t_cur  = t_next - STEP

    if t_cur not in time_to_j:
        continue

    j_orig = time_to_j[t_cur]

    x_past_orig = x_past_train[j_orig].cpu().numpy()
    y_past_orig = y_past_train[j_orig].cpu().numpy()
    y_ref_orig  = y_ref_train_seq[j_orig].cpu().numpy()

    u_past_orig = x_past_orig[:, -1]
    u_past_aug  = x_past_aug[:, -1]

    x_future_orig  = x_future_train[j_orig:j_orig+1].to(device)
    y_ref_orig_t   = y_ref_train_seq[j_orig:j_orig+1].to(device)
    y_const_orig_t = y_const_train_seq[j_orig:j_orig+1].to(device)
    x_past_orig_t  = x_past_train[j_orig:j_orig+1].to(device)
    y_past_orig_t  = y_past_train[j_orig:j_orig+1].to(device)

    with torch.no_grad():
        policy_in_past   = torch.cat((x_past_orig_t, y_past_orig_t), dim=2)
        policy_in_future = torch.cat((x_future_orig, y_ref_orig_t, y_const_orig_t), dim=2)
        u_seq_orig = model((policy_in_past, policy_in_future))
    u_future_orig = u_seq_orig.cpu().numpy()[0, :, 0]

    # 2nd 샘플
    has_2nd = t_next in time_to_2nd
    if has_2nd:
        s2 = time_to_2nd[t_next]
        x_past_2nd = s2[0].cpu().numpy() if torch.is_tensor(s2[0]) else np.asarray(s2[0])
        y_past_2nd = s2[1].cpu().numpy() if torch.is_tensor(s2[1]) else np.asarray(s2[1])
        y_ref_2nd  = s2[3].cpu().numpy() if torch.is_tensor(s2[3]) else np.asarray(s2[3])
        u_past_2nd = x_past_2nd[:, -1]

    # =========================
    # 시간축
    # =========================
    t_orig_past   = np.arange(0, 50)
    t_orig_future = np.arange(50, 100)
    t_aug_past    = np.arange(50, 100)
    t_2nd_past    = np.arange(100, 150)

    depth_lower_const = 0.1423
    depth_upper_const = 0.4126

    fig, axes = plt.subplots(3, 2, figsize=(14, 5),
                             gridspec_kw={'hspace': 0.1, 'wspace': 0.3})

    # ══════════════════════════════
    # 좌측: 1st (x축 0~100)
    # ══════════════════════════════

    # TEMP
    axes[0, 0].plot(t_orig_past,   y_past_orig[:, 0], label='orig past')
    axes[0, 0].plot(t_orig_future, y_ref_orig[:, 0],  '--', label='orig ref')
    axes[0, 0].plot(t_aug_past,    y_past_aug[:, 0],  label='aug past')
    axes[0, 0].axvline(x=50, color='gray', linestyle='--', linewidth=0.8)
    axes[0, 0].set_title("Temp (1st)")
    axes[0, 0].legend(fontsize=7)
    axes[0, 0].grid()
    axes[0, 0].tick_params(labelbottom=False)

    # DEPTH
    axes[1, 0].plot(t_orig_past, y_past_orig[:, 1], label='orig past')
    axes[1, 0].plot(t_aug_past,  y_past_aug[:, 1],  label='aug past')
    t_left = np.arange(0, 100)
    axes[1, 0].plot(t_left, [depth_lower_const]*len(t_left), 'k--', label='lower')
    axes[1, 0].plot(t_left, [depth_upper_const]*len(t_left), 'r--', label='upper')
    axes[1, 0].axvline(x=50, color='gray', linestyle='--', linewidth=0.8)
    axes[1, 0].set_title("Depth (1st)")
    axes[1, 0].legend(fontsize=7)
    axes[1, 0].grid()
    axes[1, 0].tick_params(labelbottom=False)

    # LASER
    axes[2, 0].plot(t_orig_past,   u_past_orig,         label='orig past')
    axes[2, 0].plot(t_orig_future, u_future_orig[:50],  '--', label='orig rollout')
    axes[2, 0].plot(t_aug_past,    u_past_aug,          label='aug past')
    axes[2, 0].axvline(x=50, color='gray', linestyle='--', linewidth=0.8)
    axes[2, 0].set_title("Laser (1st)")
    axes[2, 0].legend(fontsize=7)
    axes[2, 0].grid()
    axes[2, 0].set_xlabel("Time Step")

    # ══════════════════════════════
    # 우측: 2nd (x축 50~150)
    # ══════════════════════════════
    if has_2nd:
        # TEMP
        axes[0, 1].plot(t_aug_past,  y_past_aug[:, 0], label='aug past')
        axes[0, 1].plot(t_2nd_past,  y_past_2nd[:, 0], label='2nd past')
        axes[0, 1].plot(t_2nd_past,  y_ref_2nd[:, 0],  '--', label='2nd ref')
        axes[0, 1].axvline(x=100, color='gray', linestyle='--', linewidth=0.8)
        axes[0, 1].set_title("Temp (2nd)")
        axes[0, 1].legend(fontsize=7)
        axes[0, 1].grid()
        axes[0, 1].tick_params(labelbottom=False)

        # DEPTH
        axes[1, 1].plot(t_aug_past,  y_past_aug[:, 1], label='aug past')
        axes[1, 1].plot(t_2nd_past,  y_past_2nd[:, 1], label='2nd past')
        t_right = np.arange(50, 150)
        axes[1, 1].plot(t_right, [depth_lower_const]*len(t_right), 'k--', label='lower')
        axes[1, 1].plot(t_right, [depth_upper_const]*len(t_right), 'r--', label='upper')
        axes[1, 1].axvline(x=100, color='gray', linestyle='--', linewidth=0.8)
        axes[1, 1].set_title("Depth (2nd)")
        axes[1, 1].legend(fontsize=7)
        axes[1, 1].grid()
        axes[1, 1].tick_params(labelbottom=False)

        # LASER
        axes[2, 1].plot(t_aug_past,  u_past_aug,  label='aug past')
        axes[2, 1].plot(t_2nd_past,  u_past_2nd,  label='2nd past')
        axes[2, 1].axvline(x=100, color='gray', linestyle='--', linewidth=0.8)
        axes[2, 1].set_title("Laser (2nd)")
        axes[2, 1].legend(fontsize=7)
        axes[2, 1].grid()
        axes[2, 1].set_xlabel("Time Step")
    else:
        for r in range(3):
            axes[r, 1].axis('off')

    plt.suptitle(f"Sample {i} | loc-z={loc_z:.3f} | t_cur={t_cur} | has_2nd={has_2nd}")
    plt.tight_layout()
    plt.show()

NameError: name 'aug_samples_2nd' is not defined

In [25]:
aug_samples_3rd = []

for s in aug_samples_2nd:
    x_past_s   = s[0].unsqueeze(0).to(device)
    y_past_s   = s[1].unsqueeze(0).to(device)
    x_future_s = s[2].unsqueeze(0).to(device)
    y_ref_s    = s[3].unsqueeze(0).to(device)
    y_const_s  = s[4].unsqueeze(0).to(device)
    t_cur      = int(s[5].item())
    t_next     = t_cur + 50

    with torch.no_grad():
        policy_in_past   = torch.cat((x_past_s, y_past_s), dim=2)
        policy_in_future = torch.cat((x_future_s, y_ref_s, y_const_s), dim=2)
        u_seq = model((policy_in_past, policy_in_future))

        x_future_tide = torch.cat((x_future_s, u_seq), dim=2)
        past_cov      = torch.cat((y_past_s, x_past_s), dim=2)
        tide_pred     = TiDE((past_cov, x_future_tide, None))

    u_step    = u_seq[:, :50, :]
    yhat_step = tide_pred.median(dim=-1).values[:, :50, :]
    x_step    = torch.cat((x_future_s[:, :50, :], u_step), dim=2)

    x_past_new = torch.cat((x_past_s[:, 50:, :], x_step), dim=1)
    y_past_new = torch.cat((y_past_s[:, 50:, :], yhat_step), dim=1)

    if t_next not in time_to_j:
        x_future_next = s[2].detach().cpu()
        y_ref_next    = s[3].detach().cpu()
        y_const_next  = s[4].detach().cpu()
    else:
        j_next        = time_to_j[t_next]
        x_future_next = x_future_train[j_next].detach().cpu()
        y_ref_next    = y_ref_train_seq[j_next].detach().cpu()
        y_const_next  = y_const_train_seq[j_next].detach().cpu()

    aug_samples_3rd.append((
        x_past_new.squeeze(0).detach().cpu(),
        y_past_new.squeeze(0).detach().cpu(),
        x_future_next,
        y_ref_next,
        y_const_next,
        torch.tensor(t_next, dtype=torch.long)
    ))

print(f"✅ 3rd rollout 샘플 개수: {len(aug_samples_3rd)}")

NameError: name 'aug_samples_2nd' is not defined

In [26]:
import numpy as np
import matplotlib.pyplot as plt
import torch

STEP = 50

# 1:1 매핑
time_to_2nd = {int(s[5].item()) - 50: s for s in aug_samples_2nd}
time_to_3rd = {int(s[5].item()) - 50: s for s in aug_samples_3rd}

# =========================
# 1. loc-z 기준 정렬
# =========================
samples_with_locz = []
for sample in aug_samples:
    x_past_aug_np = sample[0].cpu().numpy() if torch.is_tensor(sample[0]) else np.asarray(sample[0])
    loc_z = x_past_aug_np[0, 0]
    samples_with_locz.append((loc_z, sample))
samples_with_locz.sort(key=lambda x: x[0])

# =========================
# 2. plot
# =========================
for i, (loc_z, sample) in enumerate(samples_with_locz):

    x_past_aug, y_past_aug, x_future_next, y_ref_next, y_const_next, time_idx_next = sample

    x_past_aug = x_past_aug.cpu().numpy() if torch.is_tensor(x_past_aug) else np.asarray(x_past_aug)
    y_past_aug = y_past_aug.cpu().numpy() if torch.is_tensor(y_past_aug) else np.asarray(y_past_aug)
    y_ref_next = y_ref_next.cpu().numpy() if torch.is_tensor(y_ref_next) else np.asarray(y_ref_next)

    t_next = int(time_idx_next.item())
    t_cur  = t_next - STEP

    if t_cur not in time_to_j:
        continue

    j_orig = time_to_j[t_cur]

    x_past_orig = x_past_train[j_orig].cpu().numpy()
    y_past_orig = y_past_train[j_orig].cpu().numpy()
    y_ref_orig  = y_ref_train_seq[j_orig].cpu().numpy()

    u_past_orig = x_past_orig[:, -1]
    u_past_aug  = x_past_aug[:, -1]

    x_future_orig  = x_future_train[j_orig:j_orig+1].to(device)
    y_ref_orig_t   = y_ref_train_seq[j_orig:j_orig+1].to(device)
    y_const_orig_t = y_const_train_seq[j_orig:j_orig+1].to(device)
    x_past_orig_t  = x_past_train[j_orig:j_orig+1].to(device)
    y_past_orig_t  = y_past_train[j_orig:j_orig+1].to(device)

    with torch.no_grad():
        policy_in_past   = torch.cat((x_past_orig_t, y_past_orig_t), dim=2)
        policy_in_future = torch.cat((x_future_orig, y_ref_orig_t, y_const_orig_t), dim=2)
        u_seq_orig = model((policy_in_past, policy_in_future))
    u_future_orig = u_seq_orig.cpu().numpy()[0, :, 0]

    # 2nd 샘플
    has_2nd = t_next in time_to_2nd
    if has_2nd:
        s2 = time_to_2nd[t_next]
        x_past_2nd = s2[0].cpu().numpy() if torch.is_tensor(s2[0]) else np.asarray(s2[0])
        y_past_2nd = s2[1].cpu().numpy() if torch.is_tensor(s2[1]) else np.asarray(s2[1])
        y_ref_2nd  = s2[3].cpu().numpy() if torch.is_tensor(s2[3]) else np.asarray(s2[3])
        u_past_2nd = x_past_2nd[:, -1]

        # 3rd 샘플 (2nd time_idx 기준)
        t_next_2nd = int(s2[5].item())
        has_3rd = t_next_2nd in time_to_3rd
        if has_3rd:
            s3 = time_to_3rd[t_next_2nd]
            x_past_3rd = s3[0].cpu().numpy() if torch.is_tensor(s3[0]) else np.asarray(s3[0])
            y_past_3rd = s3[1].cpu().numpy() if torch.is_tensor(s3[1]) else np.asarray(s3[1])
            y_ref_3rd  = s3[3].cpu().numpy() if torch.is_tensor(s3[3]) else np.asarray(s3[3])
            u_past_3rd = x_past_3rd[:, -1]
    else:
        has_3rd = False

    # =========================
    # 시간축
    # =========================
    t_orig_past   = np.arange(0, 50)
    t_orig_future = np.arange(50, 100)
    t_aug_past    = np.arange(50, 100)
    t_2nd_past    = np.arange(100, 150)
    t_3rd_past    = np.arange(150, 200)

    depth_lower_const = 0.1423
    depth_upper_const = 0.4126

    fig, axes = plt.subplots(3, 3, figsize=(18, 5),
                             gridspec_kw={'hspace': 0.1, 'wspace': 0.3})

    # ══════════════════════════════
    # 좌측: 1st (x축 0~100)
    # ══════════════════════════════

    # TEMP
    axes[0, 0].plot(t_orig_past,   y_past_orig[:, 0], label='orig past')
    axes[0, 0].plot(t_orig_future, y_ref_orig[:, 0],  '--', label='orig ref')
    axes[0, 0].plot(t_aug_past,    y_past_aug[:, 0],  label='aug past')
    axes[0, 0].axvline(x=50, color='gray', linestyle='--', linewidth=0.8)
    axes[0, 0].set_title("Temp (1st)")
    axes[0, 0].legend(fontsize=7)
    axes[0, 0].grid()
    axes[0, 0].tick_params(labelbottom=False)

    # DEPTH
    axes[1, 0].plot(t_orig_past, y_past_orig[:, 1], label='orig past')
    axes[1, 0].plot(t_aug_past,  y_past_aug[:, 1],  label='aug past')
    t_left = np.arange(0, 100)
    axes[1, 0].plot(t_left, [depth_lower_const]*len(t_left), 'k--', label='lower')
    axes[1, 0].plot(t_left, [depth_upper_const]*len(t_left), 'r--', label='upper')
    axes[1, 0].axvline(x=50, color='gray', linestyle='--', linewidth=0.8)
    axes[1, 0].set_title("Depth (1st)")
    axes[1, 0].legend(fontsize=7)
    axes[1, 0].grid()
    axes[1, 0].tick_params(labelbottom=False)

    # LASER
    axes[2, 0].plot(t_orig_past,   u_past_orig,        label='orig past')
    axes[2, 0].plot(t_orig_future, u_future_orig[:50], '--', label='orig rollout')
    axes[2, 0].plot(t_aug_past,    u_past_aug,         label='aug past')
    axes[2, 0].axvline(x=50, color='gray', linestyle='--', linewidth=0.8)
    axes[2, 0].set_title("Laser (1st)")
    axes[2, 0].legend(fontsize=7)
    axes[2, 0].grid()
    axes[2, 0].set_xlabel("Time Step")

    # ══════════════════════════════
    # 중간: 2nd (x축 50~150)
    # ══════════════════════════════
    if has_2nd:
        axes[0, 1].plot(t_aug_past,  y_past_aug[:, 0], label='aug past')
        axes[0, 1].plot(t_2nd_past,  y_past_2nd[:, 0], label='2nd past')
        axes[0, 1].plot(t_2nd_past,  y_ref_2nd[:, 0],  '--', label='2nd ref')
        axes[0, 1].axvline(x=100, color='gray', linestyle='--', linewidth=0.8)
        axes[0, 1].set_title("Temp (2nd)")
        axes[0, 1].legend(fontsize=7)
        axes[0, 1].grid()
        axes[0, 1].tick_params(labelbottom=False)

        axes[1, 1].plot(t_aug_past,  y_past_aug[:, 1], label='aug past')
        axes[1, 1].plot(t_2nd_past,  y_past_2nd[:, 1], label='2nd past')
        t_mid = np.arange(50, 150)
        axes[1, 1].plot(t_mid, [depth_lower_const]*len(t_mid), 'k--', label='lower')
        axes[1, 1].plot(t_mid, [depth_upper_const]*len(t_mid), 'r--', label='upper')
        axes[1, 1].axvline(x=100, color='gray', linestyle='--', linewidth=0.8)
        axes[1, 1].set_title("Depth (2nd)")
        axes[1, 1].legend(fontsize=7)
        axes[1, 1].grid()
        axes[1, 1].tick_params(labelbottom=False)

        axes[2, 1].plot(t_aug_past,  u_past_aug,  label='aug past')
        axes[2, 1].plot(t_2nd_past,  u_past_2nd,  label='2nd past')
        axes[2, 1].axvline(x=100, color='gray', linestyle='--', linewidth=0.8)
        axes[2, 1].set_title("Laser (2nd)")
        axes[2, 1].legend(fontsize=7)
        axes[2, 1].grid()
        axes[2, 1].set_xlabel("Time Step")
    else:
        for r in range(3):
            axes[r, 1].axis('off')

    # ══════════════════════════════
    # 우측: 3rd (x축 100~200)
    # ══════════════════════════════
    if has_3rd:
        axes[0, 2].plot(t_2nd_past,  y_past_2nd[:, 0], label='2nd past')
        axes[0, 2].plot(t_3rd_past,  y_past_3rd[:, 0], label='3rd past')
        axes[0, 2].plot(t_3rd_past,  y_ref_3rd[:, 0],  '--', label='3rd ref')
        axes[0, 2].axvline(x=150, color='gray', linestyle='--', linewidth=0.8)
        axes[0, 2].set_title("Temp (3rd)")
        axes[0, 2].legend(fontsize=7)
        axes[0, 2].grid()
        axes[0, 2].tick_params(labelbottom=False)

        axes[1, 2].plot(t_2nd_past,  y_past_2nd[:, 1], label='2nd past')
        axes[1, 2].plot(t_3rd_past,  y_past_3rd[:, 1], label='3rd past')
        t_right = np.arange(100, 200)
        axes[1, 2].plot(t_right, [depth_lower_const]*len(t_right), 'k--', label='lower')
        axes[1, 2].plot(t_right, [depth_upper_const]*len(t_right), 'r--', label='upper')
        axes[1, 2].axvline(x=150, color='gray', linestyle='--', linewidth=0.8)
        axes[1, 2].set_title("Depth (3rd)")
        axes[1, 2].legend(fontsize=7)
        axes[1, 2].grid()
        axes[1, 2].tick_params(labelbottom=False)

        axes[2, 2].plot(t_2nd_past,  u_past_2nd,  label='2nd past')
        axes[2, 2].plot(t_3rd_past,  u_past_3rd,  label='3rd past')
        axes[2, 2].axvline(x=150, color='gray', linestyle='--', linewidth=0.8)
        axes[2, 2].set_title("Laser (3rd)")
        axes[2, 2].legend(fontsize=7)
        axes[2, 2].grid()
        axes[2, 2].set_xlabel("Time Step")
    else:
        for r in range(3):
            axes[r, 2].axis('off')

    plt.suptitle(f"Sample {i} | loc-z={loc_z:.3f} | t_cur={t_cur} | has_2nd={has_2nd} | has_3rd={has_3rd}")
    plt.tight_layout()
    plt.show()

NameError: name 'aug_samples_2nd' is not defined

In [27]:
count_fallback = 0
for s in aug_samples_2nd:
    t_cur = int(s[5].item())
    t_next = t_cur + 50
    if t_next not in time_to_j:
        count_fallback += 1
print(f"fallback 비율: {count_fallback}/{len(aug_samples_2nd)}")

# 15번 샘플 기준으로 직접 추적
s = aug_samples_2nd[15]
t_cur = int(s[5].item())
t_next = t_cur + 50
print(f"t_cur: {t_cur}, t_next: {t_next}")
print(f"t_next in time_to_j: {t_next in time_to_j}")
if t_next in time_to_j:
    j_next = time_to_j[t_next]
    print(f"j_next: {j_next}")
    print(f"y_ref_train_seq[j_next][:5]: {y_ref_train_seq[j_next][:5, 0]}")
print(f"aug_samples_3rd[15][3][:5]: {aug_samples_3rd[15][3][:5, 0]}")

NameError: name 'aug_samples_2nd' is not defined

In [28]:
aug_samples_4th = []

for s in aug_samples_3rd:
    x_past_s   = s[0].unsqueeze(0).to(device)
    y_past_s   = s[1].unsqueeze(0).to(device)
    x_future_s = s[2].unsqueeze(0).to(device)
    y_ref_s    = s[3].unsqueeze(0).to(device)
    y_const_s  = s[4].unsqueeze(0).to(device)
    t_cur      = int(s[5].item())
    t_next     = t_cur + 50

    with torch.no_grad():
        policy_in_past   = torch.cat((x_past_s, y_past_s), dim=2)
        policy_in_future = torch.cat((x_future_s, y_ref_s, y_const_s), dim=2)
        u_seq = model((policy_in_past, policy_in_future))

        x_future_tide = torch.cat((x_future_s, u_seq), dim=2)
        past_cov      = torch.cat((y_past_s, x_past_s), dim=2)
        tide_pred     = TiDE((past_cov, x_future_tide, None))

    u_step    = u_seq[:, :50, :]
    yhat_step = tide_pred.median(dim=-1).values[:, :50, :]
    x_step    = torch.cat((x_future_s[:, :50, :], u_step), dim=2)

    x_past_new = torch.cat((x_past_s[:, 50:, :], x_step), dim=1)
    y_past_new = torch.cat((y_past_s[:, 50:, :], yhat_step), dim=1)

    if t_next not in time_to_j:
        x_future_next = s[2].detach().cpu()
        y_ref_next    = s[3].detach().cpu()
        y_const_next  = s[4].detach().cpu()
    else:
        j_next        = time_to_j[t_next]
        x_future_next = x_future_train[j_next].detach().cpu()
        y_ref_next    = y_ref_train_seq[j_next].detach().cpu()
        y_const_next  = y_const_train_seq[j_next].detach().cpu()

    aug_samples_4th.append((
        x_past_new.squeeze(0).detach().cpu(),
        y_past_new.squeeze(0).detach().cpu(),
        x_future_next,
        y_ref_next,
        y_const_next,
        torch.tensor(t_next, dtype=torch.long)
    ))

print(f"✅ 4th rollout 샘플 개수: {len(aug_samples_4th)}")

✅ 4th rollout 샘플 개수: 0


In [29]:
import numpy as np
import matplotlib.pyplot as plt
import torch

STEP = 50

# 1:1 매핑
time_to_2nd = {int(s[5].item()) - 50: s for s in aug_samples_2nd}
time_to_3rd = {int(s[5].item()) - 50: s for s in aug_samples_3rd}
time_to_4th = {int(s[5].item()) - 50: s for s in aug_samples_4th}

# =========================
# 1. loc-z 기준 정렬
# =========================
samples_with_locz = []
for sample in aug_samples:
    x_past_aug_np = sample[0].cpu().numpy() if torch.is_tensor(sample[0]) else np.asarray(sample[0])
    loc_z = x_past_aug_np[0, 0]
    samples_with_locz.append((loc_z, sample))
samples_with_locz.sort(key=lambda x: x[0])

# =========================
# 2. plot
# =========================
for i, (loc_z, sample) in enumerate(samples_with_locz):

    x_past_aug, y_past_aug, x_future_next, y_ref_next, y_const_next, time_idx_next = sample

    x_past_aug = x_past_aug.cpu().numpy() if torch.is_tensor(x_past_aug) else np.asarray(x_past_aug)
    y_past_aug = y_past_aug.cpu().numpy() if torch.is_tensor(y_past_aug) else np.asarray(y_past_aug)
    y_ref_next = y_ref_next.cpu().numpy() if torch.is_tensor(y_ref_next) else np.asarray(y_ref_next)

    t_next = int(time_idx_next.item())
    t_cur  = t_next - STEP

    if t_cur not in time_to_j:
        continue

    j_orig = time_to_j[t_cur]

    x_past_orig = x_past_train[j_orig].cpu().numpy()
    y_past_orig = y_past_train[j_orig].cpu().numpy()
    y_ref_orig  = y_ref_train_seq[j_orig].cpu().numpy()

    u_past_orig = x_past_orig[:, -1]
    u_past_aug  = x_past_aug[:, -1]

    x_future_orig  = x_future_train[j_orig:j_orig+1].to(device)
    y_ref_orig_t   = y_ref_train_seq[j_orig:j_orig+1].to(device)
    y_const_orig_t = y_const_train_seq[j_orig:j_orig+1].to(device)
    x_past_orig_t  = x_past_train[j_orig:j_orig+1].to(device)
    y_past_orig_t  = y_past_train[j_orig:j_orig+1].to(device)

    with torch.no_grad():
        policy_in_past   = torch.cat((x_past_orig_t, y_past_orig_t), dim=2)
        policy_in_future = torch.cat((x_future_orig, y_ref_orig_t, y_const_orig_t), dim=2)
        u_seq_orig = model((policy_in_past, policy_in_future))
    u_future_orig = u_seq_orig.cpu().numpy()[0, :, 0]

    # 2nd
    has_2nd = t_next in time_to_2nd
    if has_2nd:
        s2 = time_to_2nd[t_next]
        x_past_2nd = s2[0].cpu().numpy() if torch.is_tensor(s2[0]) else np.asarray(s2[0])
        y_past_2nd = s2[1].cpu().numpy() if torch.is_tensor(s2[1]) else np.asarray(s2[1])
        y_ref_2nd  = s2[3].cpu().numpy() if torch.is_tensor(s2[3]) else np.asarray(s2[3])
        u_past_2nd = x_past_2nd[:, -1]
        t_next_2nd = int(s2[5].item())

        # 3rd
        has_3rd = t_next_2nd in time_to_3rd
        if has_3rd:
            s3 = time_to_3rd[t_next_2nd]
            x_past_3rd = s3[0].cpu().numpy() if torch.is_tensor(s3[0]) else np.asarray(s3[0])
            y_past_3rd = s3[1].cpu().numpy() if torch.is_tensor(s3[1]) else np.asarray(s3[1])
            y_ref_3rd  = s3[3].cpu().numpy() if torch.is_tensor(s3[3]) else np.asarray(s3[3])
            u_past_3rd = x_past_3rd[:, -1]
            t_next_3rd = int(s3[5].item())

            # 4th
            has_4th = t_next_3rd in time_to_4th
            if has_4th:
                s4 = time_to_4th[t_next_3rd]
                x_past_4th = s4[0].cpu().numpy() if torch.is_tensor(s4[0]) else np.asarray(s4[0])
                y_past_4th = s4[1].cpu().numpy() if torch.is_tensor(s4[1]) else np.asarray(s4[1])
                y_ref_4th  = s4[3].cpu().numpy() if torch.is_tensor(s4[3]) else np.asarray(s4[3])
                u_past_4th = x_past_4th[:, -1]
        else:
            has_4th = False
    else:
        has_3rd = False
        has_4th = False

    # =========================
    # 시간축
    # =========================
    t_orig_past   = np.arange(0, 50)
    t_orig_future = np.arange(50, 100)
    t_aug_past    = np.arange(50, 100)
    t_2nd_past    = np.arange(100, 150)
    t_3rd_past    = np.arange(150, 200)
    t_4th_past    = np.arange(200, 250)

    depth_lower_const = 0.1423
    depth_upper_const = 0.4126

    fig, axes = plt.subplots(3, 4, figsize=(22, 5),
                             gridspec_kw={'hspace': 0.1, 'wspace': 0.3})

    # ══════════════════════════════
    # col 0: 1st (x축 0~100)
    # ══════════════════════════════
    axes[0, 0].plot(t_orig_past,   y_past_orig[:, 0], label='orig past')
    axes[0, 0].plot(t_orig_future, y_ref_orig[:, 0],  '--', label='orig ref')
    axes[0, 0].plot(t_aug_past,    y_past_aug[:, 0],  label='aug past')
    axes[0, 0].axvline(x=50, color='gray', linestyle='--', linewidth=0.8)
    axes[0, 0].set_title("Temp (1st)")
    axes[0, 0].legend(fontsize=7)
    axes[0, 0].grid()
    axes[0, 0].tick_params(labelbottom=False)

    axes[1, 0].plot(t_orig_past, y_past_orig[:, 1], label='orig past')
    axes[1, 0].plot(t_aug_past,  y_past_aug[:, 1],  label='aug past')
    t_left = np.arange(0, 100)
    axes[1, 0].plot(t_left, [depth_lower_const]*len(t_left), 'k--', label='lower')
    axes[1, 0].plot(t_left, [depth_upper_const]*len(t_left), 'r--', label='upper')
    axes[1, 0].axvline(x=50, color='gray', linestyle='--', linewidth=0.8)
    axes[1, 0].set_title("Depth (1st)")
    axes[1, 0].legend(fontsize=7)
    axes[1, 0].grid()
    axes[1, 0].tick_params(labelbottom=False)

    axes[2, 0].plot(t_orig_past,   u_past_orig,        label='orig past')
    axes[2, 0].plot(t_orig_future, u_future_orig[:50], '--', label='orig rollout')
    axes[2, 0].plot(t_aug_past,    u_past_aug,         label='aug past')
    axes[2, 0].axvline(x=50, color='gray', linestyle='--', linewidth=0.8)
    axes[2, 0].set_title("Laser (1st)")
    axes[2, 0].legend(fontsize=7)
    axes[2, 0].grid()
    axes[2, 0].set_xlabel("Time Step")

    # ══════════════════════════════
    # col 1: 2nd (x축 50~150)
    # ══════════════════════════════
    if has_2nd:
        axes[0, 1].plot(t_aug_past,  y_past_aug[:, 0], label='aug past')
        axes[0, 1].plot(t_2nd_past,  y_past_2nd[:, 0], label='2nd past')
        axes[0, 1].plot(t_2nd_past,  y_ref_2nd[:, 0],  '--', label='2nd ref')
        axes[0, 1].axvline(x=100, color='gray', linestyle='--', linewidth=0.8)
        axes[0, 1].set_title("Temp (2nd)")
        axes[0, 1].legend(fontsize=7)
        axes[0, 1].grid()
        axes[0, 1].tick_params(labelbottom=False)

        axes[1, 1].plot(t_aug_past,  y_past_aug[:, 1], label='aug past')
        axes[1, 1].plot(t_2nd_past,  y_past_2nd[:, 1], label='2nd past')
        t_mid1 = np.arange(50, 150)
        axes[1, 1].plot(t_mid1, [depth_lower_const]*len(t_mid1), 'k--', label='lower')
        axes[1, 1].plot(t_mid1, [depth_upper_const]*len(t_mid1), 'r--', label='upper')
        axes[1, 1].axvline(x=100, color='gray', linestyle='--', linewidth=0.8)
        axes[1, 1].set_title("Depth (2nd)")
        axes[1, 1].legend(fontsize=7)
        axes[1, 1].grid()
        axes[1, 1].tick_params(labelbottom=False)

        axes[2, 1].plot(t_aug_past,  u_past_aug,  label='aug past')
        axes[2, 1].plot(t_2nd_past,  u_past_2nd,  label='2nd past')
        axes[2, 1].axvline(x=100, color='gray', linestyle='--', linewidth=0.8)
        axes[2, 1].set_title("Laser (2nd)")
        axes[2, 1].legend(fontsize=7)
        axes[2, 1].grid()
        axes[2, 1].set_xlabel("Time Step")
    else:
        for r in range(3):
            axes[r, 1].axis('off')

    # ══════════════════════════════
    # col 2: 3rd (x축 100~200)
    # ══════════════════════════════
    if has_3rd:
        axes[0, 2].plot(t_2nd_past,  y_past_2nd[:, 0], label='2nd past')
        axes[0, 2].plot(t_3rd_past,  y_past_3rd[:, 0], label='3rd past')
        axes[0, 2].plot(t_3rd_past,  y_ref_3rd[:, 0],  '--', label='3rd ref')
        axes[0, 2].axvline(x=150, color='gray', linestyle='--', linewidth=0.8)
        axes[0, 2].set_title("Temp (3rd)")
        axes[0, 2].legend(fontsize=7)
        axes[0, 2].grid()
        axes[0, 2].tick_params(labelbottom=False)

        axes[1, 2].plot(t_2nd_past,  y_past_2nd[:, 1], label='2nd past')
        axes[1, 2].plot(t_3rd_past,  y_past_3rd[:, 1], label='3rd past')
        t_mid2 = np.arange(100, 200)
        axes[1, 2].plot(t_mid2, [depth_lower_const]*len(t_mid2), 'k--', label='lower')
        axes[1, 2].plot(t_mid2, [depth_upper_const]*len(t_mid2), 'r--', label='upper')
        axes[1, 2].axvline(x=150, color='gray', linestyle='--', linewidth=0.8)
        axes[1, 2].set_title("Depth (3rd)")
        axes[1, 2].legend(fontsize=7)
        axes[1, 2].grid()
        axes[1, 2].tick_params(labelbottom=False)

        axes[2, 2].plot(t_2nd_past,  u_past_2nd,  label='2nd past')
        axes[2, 2].plot(t_3rd_past,  u_past_3rd,  label='3rd past')
        axes[2, 2].axvline(x=150, color='gray', linestyle='--', linewidth=0.8)
        axes[2, 2].set_title("Laser (3rd)")
        axes[2, 2].legend(fontsize=7)
        axes[2, 2].grid()
        axes[2, 2].set_xlabel("Time Step")
    else:
        for r in range(3):
            axes[r, 2].axis('off')

    # ══════════════════════════════
    # col 3: 4th (x축 150~250)
    # ══════════════════════════════
    if has_4th:
        axes[0, 3].plot(t_3rd_past,  y_past_3rd[:, 0], label='3rd past')
        axes[0, 3].plot(t_4th_past,  y_past_4th[:, 0], label='4th past')
        axes[0, 3].plot(t_4th_past,  y_ref_4th[:, 0],  '--', label='4th ref')
        axes[0, 3].axvline(x=200, color='gray', linestyle='--', linewidth=0.8)
        axes[0, 3].set_title("Temp (4th)")
        axes[0, 3].legend(fontsize=7)
        axes[0, 3].grid()
        axes[0, 3].tick_params(labelbottom=False)

        axes[1, 3].plot(t_3rd_past,  y_past_3rd[:, 1], label='3rd past')
        axes[1, 3].plot(t_4th_past,  y_past_4th[:, 1], label='4th past')
        t_right = np.arange(150, 250)
        axes[1, 3].plot(t_right, [depth_lower_const]*len(t_right), 'k--', label='lower')
        axes[1, 3].plot(t_right, [depth_upper_const]*len(t_right), 'r--', label='upper')
        axes[1, 3].axvline(x=200, color='gray', linestyle='--', linewidth=0.8)
        axes[1, 3].set_title("Depth (4th)")
        axes[1, 3].legend(fontsize=7)
        axes[1, 3].grid()
        axes[1, 3].tick_params(labelbottom=False)

        axes[2, 3].plot(t_3rd_past,  u_past_3rd,  label='3rd past')
        axes[2, 3].plot(t_4th_past,  u_past_4th,  label='4th past')
        axes[2, 3].axvline(x=200, color='gray', linestyle='--', linewidth=0.8)
        axes[2, 3].set_title("Laser (4th)")
        axes[2, 3].legend(fontsize=7)
        axes[2, 3].grid()
        axes[2, 3].set_xlabel("Time Step")
    else:
        for r in range(3):
            axes[r, 3].axis('off')

    plt.suptitle(f"Sample {i} | loc-z={loc_z:.3f} | t_cur={t_cur} | has_2nd={has_2nd} | has_3rd={has_3rd} | has_4th={has_4th}")
    plt.tight_layout()
    plt.show()

NameError: name 'aug_samples_2nd' is not defined